# 05 â€” Strategy comparison dashboard

Capstone notebook that runs all three strategies against the full 2015â€“2024
universe, generates publication-quality comparison figures, and exports them
to `analysis/figures/` as PNG and PDF.

**Figures produced:**
| File | Description |
|------|-------------|
| `01_equity_curves` | Normalized portfolio value (base=100) for all strategies vs SPY |
| `02_drawdowns` | Drawdown depth comparison across strategies |
| `03_metrics_comparison` | Side-by-side 8-metric performance bar charts |
| `04_rolling_metrics` | 63-day rolling return, Sharpe, and max drawdown |
| `05_correlation_matrix` | Pearson return-correlation heatmap |
| `06_monthly_returns_*` | Monthly return calendar heatmap per strategy |
| `07_trade_analysis_*` | Trade cost and size breakdown per strategy |
| `08_positions_*` | Position weight concentration per strategy |
| `09_combined_portfolio` | Equal-weight combined portfolio vs individuals |
| `10_combined_monthly_returns` | Combined portfolio monthly heatmap |

In [1]:
%matplotlib inline
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on sys.path when running from notebooks/
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from analysis import (
    plot_equity_curves,
    plot_drawdowns,
    plot_metrics_comparison,
    plot_rolling_metrics,
    plot_correlation_matrix,
    plot_monthly_returns_heatmap,
    plot_trade_analysis,
    plot_position_concentration,
    save_figure,
)
from backtester import (
    DataLoader, Backtester, BacktestResult, compute_metrics,
)
from strategies import (
    MomentumStrategy,
    MeanReversionStrategy,
    MLSignalStrategy,
)
from config import (
    DEFAULT_TICKERS, BENCHMARK_TICKER,
    START_DATE, END_DATE, INITIAL_CAPITAL,
)

FIGURES_DIR = pathlib.Path("../analysis/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {FIGURES_DIR.resolve()}")

Figures will be saved to: C:\Users\fariz\Desktop\Algorithmic-trading-backtester\analysis\figures


In [2]:
all_tickers = DEFAULT_TICKERS + [BENCHMARK_TICKER]
loader = DataLoader(all_tickers, START_DATE, END_DATE)
prices = loader.load_wide()
print(f"Loaded {prices.shape[1]} tickers x {prices.shape[0]} trading days")
print(f"Date range: {prices.index[0].date()} -> {prices.index[-1].date()}")
prices.head()

Loading AAPL from cache...
Loading MSFT from cache...
Loading NVDA from cache...
Loading AMZN from cache...
Loading GOOGL from cache...
Loading META from cache...
Loading BRK-B from cache...
Loading LLY from cache...
Loading AVGO from cache...
Loading JPM from cache...
Loading UNH from cache...
Loading XOM from cache...
Loading TSLA from cache...
Loading V from cache...
Loading MA from cache...
Loading PG from cache...
Loading JNJ from cache...
Loading COST from cache...
Loading HD from cache...
Loading MRK from cache...
Loading SPY from cache...
Loaded 21 tickers x 2515 trading days
Date range: 2015-01-02 -> 2024-12-30


,AAPL,AMZN,AVGO,BRK-B,COST,GOOGL,HD,JNJ,JPM,LLY,...,META,MRK,MSFT,NVDA,PG,SPY,TSLA,UNH,V,XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,24.214893,15.4260,7.544514,149.169998,113.980286,26.260456,79.371254,76.548653,46.274315,57.380966,...,77.839165,38.484348,39.767689,0.482985,66.049377,170.124969,14.620667,83.828773,61.214729,57.533428
2015-01-05,23.532719,15.1095,7.423911,147.000000,112.684456,25.760094,77.706024,76.014000,44.837734,56.815845,...,76.588974,39.056324,39.401978,0.474828,65.735359,167.052643,14.006000,82.447983,59.863525,55.959202
2015-01-06,23.534937,14.7645,7.255064,146.839996,114.173523,25.124344,77.468124,75.640457,43.675121,57.102493,...,75.557068,40.590591,38.823673,0.460432,65.435928,165.479141,14.085333,82.281647,59.477757,55.661697
2015-01-07,23.864950,14.9210,7.451045,148.880005,116.161583,25.050459,80.123291,77.310303,43.741776,56.701180,...,75.557068,41.458656,39.316936,0.459232,65.779160,167.541229,14.063333,83.121712,60.274662,56.225700
2015-01-08,24.781891,15.0230,7.823411,151.369995,117.159660,25.137737,81.895996,77.918167,44.719242,58.044384,...,77.571259,42.293064,40.473579,0.476507,66.531380,170.514191,14.041333,87.089417,61.083092,57.161533


In [3]:
bt = Backtester(prices, config={"allow_short": False})

print("Running momentum strategy...")
mom_result = bt.run(MomentumStrategy(
    lookback_months=12, skip_months=1, n_long=5))

print("\nRunning mean reversion strategy...")
mr_result = bt.run(MeanReversionStrategy(
    bb_window=20, bb_std=1.5, rsi_oversold=45))
print(f"MeanReversion n_trades: {mr_result.metrics['n_trades']}")

print("\nRunning ML signal strategy...")
ml_result = bt.run(MLSignalStrategy(
    model_type="xgboost", min_train_years=3,
    retrain_freq_months=3))

print("\nRunning benchmark (SPY buy-and-hold)...")
spy_result = bt.run_benchmark()
spy_result.metrics = compute_metrics(spy_result)

all_results = [mom_result, mr_result, ml_result]
print("\nAll strategies complete.")

Running momentum strategy...
[Backtest] Momentum(lookback=12m, skip=1m, long=5, short=0, type=ranked) | 2015-01-02 â†’ 2024-12-30 | Trades: 183 | Final: $1,806,399.59 | Return: +1706.40%
[Backtest] BuyAndHold(SPY) | 2015-01-02 â†’ 2024-12-30 | Trades: 1 | Final: $140,514.66 | Return: +40.51%

Running mean reversion strategy...
[Backtest] MeanReversion(bb=20/1.5std, rsi=14/45/65.0, exit=mean, type=binary) | 2015-01-02 â†’ 2024-12-30 | Trades: 0 | Final: $100,000.00 | Return: +0.00%
[Backtest] BuyAndHold(SPY) | 2015-01-02 â†’ 2024-12-30 | Trades: 1 | Final: $140,514.66 | Return: +40.51%
MeanReversion n_trades: 0

Running ML signal strategy...
Walk-forward: 28 training windows, first=2018-01-03, last=2024-10-08
  [1/28] train->2018-01-03 predict 2018-01-04->2018-04-02 (1260 obs)
  [2/28] train->2018-04-03 predict 2018-04-04->2018-07-02 (1323 obs)
  [3/28] train->2018-07-03 predict 2018-07-04->2018-10-02 (1323 obs)
  [4/28] train->2018-10-03 predict 2018-10-04->2019-01-02 (1281 obs)
  [5/2

In [4]:
import subprocess
subprocess.run(["pip", "install", "jinja2"], check=True)

CompletedProcess(args=['pip', 'install', 'jinja2'], returncode=0)

In [5]:
metrics_keys = [
    "total_return_pct", "annualized_return_pct",
    "annualized_volatility_pct", "sharpe_ratio",
    "sortino_ratio", "calmar_ratio", "max_drawdown_pct",
    "max_drawdown_duration_days", "recovery_days",
    "alpha_pct", "beta", "information_ratio",
    "n_trades", "total_cost_pct", "turnover_annual_pct",
]


def safe_metric(result, key):
    val = result.metrics.get(key)
    return val  # None stays as None — pandas handles it correctly


rows = {}
for r in all_results + [spy_result]:
    rows[r.strategy_name] = {k: safe_metric(r, k) for k in metrics_keys}

summary_df = pd.DataFrame(rows).T
summary_df.index.name = "Strategy"
summary_df = summary_df.apply(pd.to_numeric, errors="coerce")

print("=== Raw metrics (copy-paste into README) ===")
print(summary_df.to_string())

# Higher-is-better metrics
higher_better = [
    "total_return_pct", "annualized_return_pct",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio",
    "alpha_pct", "information_ratio",
]
# Lower-is-better metrics
lower_better = [
    "max_drawdown_pct", "annualized_volatility_pct", "total_cost_pct",
]

try:
    import jinja2
    styled = (
        summary_df.round(2)
        .style
        .format(na_rep="—")
        .highlight_max(
            subset=[c for c in higher_better if c in summary_df.columns],
            color="lightgreen", axis=0,
        )
        .highlight_min(
            subset=[c for c in lower_better if c in summary_df.columns],
            color="lightgreen", axis=0,
        )
    )
    display(styled)
except (ImportError, AttributeError):
    display(summary_df.round(2).fillna("—"))

=== Raw metrics (copy-paste into README) ===
                                                                    total_return_pct annualized_return_pct annualized_volatility_pct sharpe_ratio sortino_ratio calmar_ratio max_drawdown_pct max_drawdown_duration_days recovery_days  alpha_pct    beta information_ratio n_trades total_cost_pct turnover_annual_pct
Strategy                                                                                                                                                                                                                                                                                                               
Momentum(lookback=12m, skip=1m, long=5, short=0, type=ranked)            1706.399589             33.638393                 24.988147       1.2967        1.6223       0.9244       -36.387582                       28.0          79.0  30.171471  4.2357            1.3194    183.0      12.043587          366.727634
MeanReversion(bb=20

TypeError: '>=' not supported between instances of 'float' and 'str'

## Figure 1 â€” Equity curves

In [6]:
fig, ax = plot_equity_curves(
    all_results, spy_result,
    title="Momentum vs Mean Reversion vs ML Signal vs SPY")
save_figure(fig, "01_equity_curves", FIGURES_DIR)
plt.show()

  Saved: ..\analysis\figures\01_equity_curves.png
  Saved: ..\analysis\figures\01_equity_curves.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\2660654067.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 2 â€” Drawdown comparison

In [7]:
fig, ax = plot_drawdowns(all_results, spy_result)
save_figure(fig, "02_drawdowns", FIGURES_DIR)
plt.show()

  Saved: ..\analysis\figures\02_drawdowns.png
  Saved: ..\analysis\figures\02_drawdowns.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1951099548.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 3 â€” Metrics comparison

In [8]:
fig, axes = plot_metrics_comparison(all_results, spy_result)
save_figure(fig, "03_metrics_comparison", FIGURES_DIR)
plt.show()

  Saved: ..\analysis\figures\03_metrics_comparison.png
  Saved: ..\analysis\figures\03_metrics_comparison.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\3623053742.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 4 â€” Rolling performance

In [9]:
fig, axes = plot_rolling_metrics(all_results, spy_result, window=63)
save_figure(fig, "04_rolling_metrics", FIGURES_DIR)
plt.show()

  Saved: ..\analysis\figures\04_rolling_metrics.png
  Saved: ..\analysis\figures\04_rolling_metrics.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1331881953.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 5 â€” Return correlation

In [10]:
fig, ax = plot_correlation_matrix(all_results, spy_result)
save_figure(fig, "05_correlation_matrix", FIGURES_DIR)
plt.show()

# Print raw correlation matrix
corr_labels = {r.strategy_name[:20]: r.returns for r in all_results + [spy_result]}
corr_df = pd.concat(corr_labels, axis=1).dropna().corr()
print("\nCorrelation matrix:")
print(corr_df.round(4).to_string())

# Find least-correlated pair
n = len(corr_df)
off_diag = [
    (corr_df.columns[i], corr_df.columns[j], corr_df.iloc[i, j])
    for i in range(n) for j in range(n) if i != j
]
min_pair = min(off_diag, key=lambda x: x[2])
print(f"\nLeast correlated pair: {min_pair[0]} <-> {min_pair[1]} (r={min_pair[2]:.3f})")
print("-> A combined portfolio of these two would benefit from meaningful diversification.")

  Saved: ..\analysis\figures\05_correlation_matrix.png
  Saved: ..\analysis\figures\05_correlation_matrix.pdf

Correlation matrix:
                      Momentum(lookback=12  MeanReversion(bb=20/  MLSignal(xgboost, fw  BuyAndHold(SPY)
Momentum(lookback=12                1.0000                   NaN                0.7822           0.7462
MeanReversion(bb=20/                   NaN                   NaN                   NaN              NaN
MLSignal(xgboost, fw                0.7822                   NaN                1.0000           0.7877
BuyAndHold(SPY)                     0.7462                   NaN                0.7877           1.0000

Least correlated pair: Momentum(lookback=12 <-> MeanReversion(bb=20/ (r=nan)
-> A combined portfolio of these two would benefit from meaningful diversification.


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1833397896.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 6 â€” Monthly returns heatmaps

In [11]:
for result in all_results:
    fig, ax = plot_monthly_returns_heatmap(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"06_monthly_returns_{name}", FIGURES_DIR)
    plt.show()

  Saved: ..\analysis\figures\06_monthly_returns_momentum.png
  Saved: ..\analysis\figures\06_monthly_returns_momentum.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1806416295.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\06_monthly_returns_meanreversion.png
  Saved: ..\analysis\figures\06_monthly_returns_meanreversion.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1806416295.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\06_monthly_returns_mlsignal.png
  Saved: ..\analysis\figures\06_monthly_returns_mlsignal.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1806416295.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 7 â€” Trade analysis

In [12]:
for result in all_results:
    fig, axes = plot_trade_analysis(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"07_trade_analysis_{name}", FIGURES_DIR)
    plt.show()

  Saved: ..\analysis\figures\07_trade_analysis_momentum.png
  Saved: ..\analysis\figures\07_trade_analysis_momentum.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\2910260873.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\07_trade_analysis_meanreversion.png
  Saved: ..\analysis\figures\07_trade_analysis_meanreversion.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\2910260873.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\07_trade_analysis_mlsignal.png
  Saved: ..\analysis\figures\07_trade_analysis_mlsignal.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\2910260873.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 8 â€” Position concentration

In [13]:
for result in all_results:
    fig, axes = plot_position_concentration(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"08_positions_{name}", FIGURES_DIR)
    plt.show()

  Saved: ..\analysis\figures\08_positions_momentum.png
  Saved: ..\analysis\figures\08_positions_momentum.pdf
  Saved: ..\analysis\figures\08_positions_meanreversion.png


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1640135387.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\08_positions_meanreversion.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1640135387.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\08_positions_mlsignal.png
  Saved: ..\analysis\figures\08_positions_mlsignal.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\1640135387.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Portfolio-level analysis

Simulate a naive equal-weight combination of all three strategies to
quantify the diversification benefit.

In [14]:
# Equal-weight average of all three strategy return streams
combined_returns = pd.concat(
    [r.returns for r in all_results], axis=1
).mean(axis=1)  # equal weight across strategies

combined_equity = INITIAL_CAPITAL * (1 + combined_returns).cumprod()

# Build a synthetic BacktestResult for the combined portfolio
combined_result = BacktestResult(
    strategy_name="Combined (equal-weight)",
    equity_curve=combined_equity,
    returns=combined_returns,
    positions=pd.DataFrame(),
    trades=pd.DataFrame(),
    metrics={},
    config={
        "initial_capital": INITIAL_CAPITAL,
        "benchmark": BENCHMARK_TICKER,
        "risk_free_rate": 0.0,
    },
)
combined_result.metrics = compute_metrics(combined_result, benchmark=spy_result)

# Diversification summary
best_sharpe = max(r.metrics.get("sharpe_ratio", 0.0) or 0.0 for r in all_results)
comb_sharpe = combined_result.metrics.get("sharpe_ratio", 0.0) or 0.0
comb_dd     = combined_result.metrics.get("max_drawdown_pct", 0.0) or 0.0
comb_alpha  = combined_result.metrics.get("alpha_pct", 0.0) or 0.0

print(f"Combined Sharpe:        {comb_sharpe:.3f}")
print(f"Combined max drawdown:  {comb_dd:.2f}%")
print(f"Combined alpha:         {comb_alpha:.2f}%")
print()
print(
    f"Diversification benefit: combined Sharpe {comb_sharpe:.3f} vs "
    f"best individual Sharpe {best_sharpe:.3f}"
)

Combined Sharpe:        1.284
Combined max drawdown:  -25.07%
Combined alpha:         16.93%

Diversification benefit: combined Sharpe 1.284 vs best individual Sharpe 1.297


## Figure 9 â€” Combined portfolio

In [15]:
fig, ax = plot_equity_curves(
    all_results + [combined_result], spy_result,
    title="Individual strategies + combined portfolio vs SPY")
save_figure(fig, "09_combined_portfolio", FIGURES_DIR)
plt.show()

fig, ax = plot_monthly_returns_heatmap(combined_result)
save_figure(fig, "10_combined_monthly_returns", FIGURES_DIR)
plt.show()

  Saved: ..\analysis\figures\09_combined_portfolio.png
  Saved: ..\analysis\figures\09_combined_portfolio.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\3236916859.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved: ..\analysis\figures\10_combined_monthly_returns.png
  Saved: ..\analysis\figures\10_combined_monthly_returns.pdf


C:\Users\fariz\AppData\Local\Temp\ipykernel_59912\3236916859.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

In [16]:
print("=" * 60)
print("=== BACKTEST SUMMARY ===")
print("=" * 60)
print(f"Period: {START_DATE} -> {END_DATE}")
print(f"Universe: {len(DEFAULT_TICKERS)} S&P 500 stocks + {BENCHMARK_TICKER} benchmark")
print(f"Initial capital: ${INITIAL_CAPITAL:,.0f}")
print()
print("Strategy performance (annualized):")

strategy_report = [
    ("Momentum",       mom_result),
    ("Mean Reversion", mr_result),
    ("ML Signal",      ml_result),
    ("Combined",       combined_result),
]

for label, r in strategy_report:
    ret    = r.metrics.get("annualized_return_pct") or float("nan")
    sharpe = r.metrics.get("sharpe_ratio")          or float("nan")
    dd     = r.metrics.get("max_drawdown_pct")      or float("nan")
    alpha  = r.metrics.get("alpha_pct")             or float("nan")
    print(
        f"  {label:<16} return={ret:.1f}%  Sharpe={sharpe:.2f}  "
        f"MaxDD={dd:.1f}%  Alpha={alpha:.1f}%"
    )

spy_ret    = spy_result.metrics.get("annualized_return_pct") or float("nan")
spy_sharpe = spy_result.metrics.get("sharpe_ratio")          or float("nan")
spy_dd     = spy_result.metrics.get("max_drawdown_pct")      or float("nan")
print(
    f"  {'SPY benchmark':<16} return={spy_ret:.1f}%  Sharpe={spy_sharpe:.2f}  "
    f"MaxDD={spy_dd:.1f}%"
)

print()
print("Correlation structure:")
strat_ret_dict = {r.strategy_name[:15]: r.returns for r in all_results}
corr_mat = pd.concat(strat_ret_dict, axis=1).dropna().corr()
print(corr_mat.round(3).to_string())

# Count exported files
n_files = sum(
    1 for p in FIGURES_DIR.glob("*")
    if p.suffix in (".png", ".pdf")
)
print(f"\nFigures exported to: {FIGURES_DIR} ({n_files} files)")

=== BACKTEST SUMMARY ===
Period: 2015-01-01 -> 2024-12-31
Universe: 20 S&P 500 stocks + SPY benchmark
Initial capital: $100,000

Strategy performance (annualized):
  Momentum         return=33.6%  Sharpe=1.30  MaxDD=-36.4%  Alpha=30.2%
  Mean Reversion   return=nan%  Sharpe=nan  MaxDD=nan%  Alpha=-3.5%
  ML Signal        return=18.4%  Sharpe=1.12  MaxDD=-36.7%  Alpha=14.9%
  Combined         return=20.4%  Sharpe=1.28  MaxDD=-25.1%  Alpha=16.9%
  SPY benchmark    return=3.5%  Sharpe=0.80  MaxDD=-9.3%

Correlation structure:
                 Momentum(lookba  MeanReversion(b  MLSignal(xgboos
Momentum(lookba            1.000              NaN            0.782
MeanReversion(b              NaN              NaN              NaN
MLSignal(xgboos            0.782              NaN            1.000

Figures exported to: ..\analysis\figures (32 files)


## Notes

- All figures are saved at 150 DPI in both PNG (for README preview) and PDF
  (for print/presentation quality).
- The ML signal strategy uses a 3-year minimum training window with quarterly
  retraining â€” this is the primary cause of its later start date.
- The combined portfolio uses a naive equal-weight average of daily returns;
  no rebalancing costs are modelled at the combination level.